In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('d:\AML_Detection_Project\HI-Small_Trans.csv')
df.head()

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/01 00:20,10,8000EBD30,10,8000EBD30,3697.34,US Dollar,3697.34,US Dollar,Reinvestment,0
1,2022/09/01 00:20,3208,8000F4580,1,8000F5340,0.01,US Dollar,0.01,US Dollar,Cheque,0
2,2022/09/01 00:00,3209,8000F4670,3209,8000F4670,14675.57,US Dollar,14675.57,US Dollar,Reinvestment,0
3,2022/09/01 00:02,12,8000F5030,12,8000F5030,2806.97,US Dollar,2806.97,US Dollar,Reinvestment,0
4,2022/09/01 00:06,10,8000F5200,10,8000F5200,36682.97,US Dollar,36682.97,US Dollar,Reinvestment,0


In [4]:
df.columns

Index(['Timestamp', 'From Bank', 'Account', 'To Bank', 'Account.1',
       'Amount Received', 'Receiving Currency', 'Amount Paid',
       'Payment Currency', 'Payment Format', 'Is Laundering'],
      dtype='object')

In [5]:
# --- Create Nodes DataFrame ---
# Combine both sender and receiver accounts
sender_nodes = df[['Account', 'From Bank']].rename(columns={'Account': 'Id', 'From Bank': 'Bank'})
receiver_nodes = df[['Account.1', 'To Bank']].rename(columns={'Account.1': 'Id', 'To Bank': 'Bank'})

nodes = pd.concat([sender_nodes, receiver_nodes]).drop_duplicates().reset_index(drop=True)
nodes['Label'] = nodes['Id']

# --- Create Edges DataFrame ---
edges = pd.DataFrame({
    'Source': df['Account'],
    'Target': df['Account.1'],
    'Weight': df['Amount Paid'],  # or 'Amount Received' depending on your use case
    'Payment Format': df['Payment Format'],
    'Is Laundering': df['Is Laundering'],
    'Start': df['Timestamp'],
    'Type': 'Directed'
})

# --- Optional: ensure proper datetime formatting for Gephi dynamic graphs ---
edges['Start'] = pd.to_datetime(edges['Start'], errors='coerce').dt.strftime('%Y-%m-%dT%H:%M:%S')

# --- Save to CSV files for Gephi ---
nodes.to_csv(r'd:\AML_Detection_Project\nodes.csv', index=False)
edges.to_csv(r'd:\AML_Detection_Project\edges.csv', index=False)

print("✅ Gephi CSVs created: nodes.csv and edges.csv")


✅ Gephi CSVs created: nodes.csv and edges.csv


In [6]:
# Count how many times each account appears
account_activity = (
    edges[['Source', 'Target']]
    .melt(value_name='Account')
    .groupby('Account')
    .size()
    .sort_values(ascending=False)
)

# Keep top 25% most active accounts
top_accounts = account_activity.head(int(0.25 * len(account_activity))).index

# Filter edges where both endpoints are in the top accounts
edges_sampled = edges[edges['Source'].isin(top_accounts) & edges['Target'].isin(top_accounts)]

nodes = pd.read_csv(r'd:\AML_Detection_Project\nodes.csv')
nodes_sampled = nodes[nodes['Id'].isin(top_accounts)]

edges_sampled.to_csv(r'd:\AML_Detection_Project\edges_top25pct.csv', index=False)
nodes_sampled.to_csv(r'd:\AML_Detection_Project\nodes_top25pct.csv', index=False)

print(f"✅ Filtered to top 25% most active accounts — {len(nodes_sampled)} nodes, {len(edges_sampled)} edges")


✅ Filtered to top 25% most active accounts — 128770 nodes, 3071408 edges


In [7]:
# Count transaction frequency per account (both as sender and receiver)
activity = (
    edges[['Source', 'Target']]
    .melt(value_name='Account')
    .groupby('Account')
    .size()
    .sort_values(ascending=False)
)

# Keep top 10% most active accounts
top_10pct = int(0.10 * len(activity))
top_accounts = activity.head(top_10pct).index

# Filter edges where both nodes are top accounts
edges_filtered = edges[edges['Source'].isin(top_accounts) & edges['Target'].isin(top_accounts)]

# Keep only nodes that appear in filtered edges
nodes = pd.read_csv(r'd:\AML_Detection_Project\nodes.csv')
nodes_filtered = nodes[nodes['Id'].isin(top_accounts)]

edges_filtered.to_csv(r'd:\AML_Detection_Project\edges_top10pct_accounts.csv', index=False)
nodes_filtered.to_csv(r'd:\AML_Detection_Project\nodes_top10pct_accounts.csv', index=False)

print(f"✅ Top 10% most active accounts kept: {len(nodes_filtered)} nodes, {len(edges_filtered)} edges")


✅ Top 10% most active accounts kept: 51508 nodes, 1191451 edges


In [8]:
# Count transaction frequency per account (both as sender and receiver)
activity = (
    edges[['Source', 'Target']]
    .melt(value_name='Account')
    .groupby('Account')
    .size()
    .sort_values(ascending=False)
)

# Keep top 5% most active accounts
top_5pct = int(0.05 * len(activity))
top_accounts = activity.head(top_5pct).index

# Filter edges where both nodes are top accounts
edges_filtered = edges[edges['Source'].isin(top_accounts) & edges['Target'].isin(top_accounts)]

# Keep only nodes that appear in filtered edges
nodes = pd.read_csv(r'd:\AML_Detection_Project\nodes.csv')
nodes_filtered = nodes[nodes['Id'].isin(top_accounts)]

edges_filtered.to_csv(r'd:\AML_Detection_Project\edges_top5pct_accounts.csv', index=False)
nodes_filtered.to_csv(r'd:\AML_Detection_Project\nodes_top5pct_accounts.csv', index=False)

print(f"✅ Top 5% most active accounts kept: {len(nodes_filtered)} nodes, {len(edges_filtered)} edges")

✅ Top 5% most active accounts kept: 25754 nodes, 546081 edges


In [9]:
# --- Load filtered edges and nodes ---
edges = pd.read_csv(r'd:\AML_Detection_Project\edges_top5pct_accounts.csv')
nodes = pd.read_csv(r'd:\AML_Detection_Project\nodes_top5pct_accounts.csv')

# --- Convert timestamp to datetime ---
edges['Start'] = pd.to_datetime(edges['Start'], errors='coerce')

# --- 1️⃣ Aggregate by DAY ---
edges['Date_Day'] = edges['Start'].dt.date

edges_day = (
    edges.groupby(['Source', 'Target', 'Date_Day'], as_index=False)
    .agg({
        'Weight': 'sum',
        'Is Laundering': lambda x: x.mode().iloc[0] if not x.mode().empty else None,
        'Payment Format': lambda x: x.mode().iloc[0] if not x.mode().empty else None
    })
)

edges_day.rename(columns={'Date_Day': 'Start'}, inplace=True)

# --- Save day-level aggregation ---
edges_day.to_csv(r'd:\AML_Detection_Project\edges_top5pct_accounts_day.csv', index=False)

# --- Compute node list for the day-aggregated graph ---
node_ids_day = pd.unique(edges_day[['Source', 'Target']].values.ravel('K'))
nodes_day = nodes[nodes['Id'].isin(node_ids_day)]
nodes_day.to_csv(r'd:\AML_Detection_Project\nodes_top5pct_accounts_day.csv', index=False)

print(f"✅ Day-level aggregation done:")
print(f"   Nodes: {len(nodes_day)}")
print(f"   Edges: {len(edges_day)}")

✅ Day-level aggregation done:
   Nodes: 25451
   Edges: 232960


In [10]:
import pandas as pd
import numpy as np

# --- 1️⃣ Load original dataset ---
df = pd.read_csv(r'd:\AML_Detection_Project\HI-Small_Trans.csv')

# --- 2️⃣ Basic cleanup ---
# Parse timestamps and ensure numeric amounts
df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')
df['Amount Paid'] = pd.to_numeric(df['Amount Paid'], errors='coerce')

# Drop rows with missing critical values
df = df.dropna(subset=['Timestamp', 'Account', 'Account.1', 'Amount Paid'])

# --- 3️⃣ Split into laundering and non-laundering ---
laundering_df = df[df['Is Laundering'] == 1]
non_laundering_df = df[df['Is Laundering'] == 0]

print(f"Laundering transactions: {len(laundering_df)}")
print(f"Non-laundering transactions: {len(non_laundering_df)}")

# --- 4️⃣ Sample non-laundering to pad up to 10,000 total ---
target_total = 10_000
needed_normals = max(0, target_total - len(laundering_df))

non_laundering_sample = non_laundering_df.sample(n=needed_normals, random_state=42)

# --- 5️⃣ Combine both ---
df_subset = pd.concat([laundering_df, non_laundering_sample]).reset_index(drop=True)

print(f"Total transactions selected: {len(df_subset)}")
print(f"Laundering proportion: {df_subset['Is Laundering'].mean():.2%}")

# --- 6️⃣ Aggregate by day (Source, Target, Date) ---
df_subset['Date'] = df_subset['Timestamp'].dt.date

edges_day = (
    df_subset.groupby(['Account', 'Account.1', 'Date'], as_index=False)
    .agg({
        'Amount Paid': 'sum',
        'Payment Format': lambda x: x.mode().iloc[0] if not x.mode().empty else None,
        'Is Laundering': lambda x: int(x.max())  # if any transaction that day is laundering
    })
)

# Rename columns for Gephi
edges_day.rename(columns={
    'Account': 'Source',
    'Account.1': 'Target',
    'Amount Paid': 'Weight',
    'Date': 'Start'
}, inplace=True)

edges_day['Type'] = 'Directed'

# --- 7️⃣ Build corresponding node list ---
node_ids = pd.unique(edges_day[['Source', 'Target']].values.ravel('K'))

# Get sender/receiver bank info if available
sender_nodes = df_subset[['Account', 'From Bank']].rename(columns={'Account': 'Id', 'From Bank': 'Bank'})
receiver_nodes = df_subset[['Account.1', 'To Bank']].rename(columns={'Account.1': 'Id', 'To Bank': 'Bank'})
nodes = pd.concat([sender_nodes, receiver_nodes]).drop_duplicates(subset='Id')
nodes['Label'] = nodes['Id']

# Flag laundering nodes
laundering_accounts = set(
    edges_day.loc[edges_day['Is Laundering'] == 1, ['Source', 'Target']].values.ravel()
)
nodes['Is_Laundering_Node'] = nodes['Id'].isin(laundering_accounts).astype(int)

# --- 8️⃣ Save for Gephi ---
edges_day.to_csv(r'd:\AML_Detection_Project\edges_subset_day.csv', index=False)
nodes.to_csv(r'd:\AML_Detection_Project\nodes_subset_day.csv', index=False)

print("✅ Gephi-ready subset created:")
print(f"   Nodes: {len(nodes)}")
print(f"   Edges (day-level): {len(edges_day)}")


Laundering transactions: 5177
Non-laundering transactions: 5073168
Total transactions selected: 10000
Laundering proportion: 51.77%
✅ Gephi-ready subset created:
   Nodes: 14645
   Edges (day-level): 9920
